In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.lead_info AS (
    SELECT id AS lead_id, name AS lead_name, display_name AS lead_display_name
    FROM (
        SELECT id, name, display_name,
            ROW_NUMBER() OVER(PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn
        FROM crm_ingestion.silver.leads_processed
        WHERE status_label = 'Coaching Client'
    )
    WHERE rn = 1
)

In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.lead_emails AS (
    SELECT
        leads_processed_id AS lead_id,
        collect_set(email) AS customer_emails
    FROM crm_ingestion.silver.leads_processed_contacts_emails
    WHERE email IS NOT NULL AND email != ''
    GROUP BY leads_processed_id
)